In [ ]:
from google.colab import drive

drive.mount("/content/drive")


# Coleta de metadados de parlamentares

Notebook Colab para clonar o repositorio, instalar dependencias e executar o coletor `coleta.parlamentares.collect`, salvando dados brutos de Camara e Senado no Google Drive.

A primeira celula monta o Drive antes de qualquer codigo do projeto. Os dados completos ficam em `MyDrive/falando_nela/data`; o clone do repositorio fica em `/content/falando_nela` e pode ser recriado a cada sessao.


In [ ]:
import os
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)

for name in ["raw", "checkpoints", "logs", "manifests", "processed"]:
    (DATA_ROOT / name).mkdir(parents=True, exist_ok=True)

print("FALANDO_NELA_DATA_ROOT=", os.environ["FALANDO_NELA_DATA_ROOT"])


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_DIR = Path("/content/falando_nela")
REPO_REF = ""  # opcional: branch, tag ou commit. Deixe vazio para usar o default remoto.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=True)
os.chdir(REPO_DIR)
print("Repo em:", Path.cwd())
subprocess.run(["git", "status", "--short"], check=True)


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


## Parametros

Use a validacao curta primeiro. Ela roda em modo producao para confirmar escrita no Drive, mas usa `--sample` para limitar poucos parlamentares por casa. A coleta completa fica protegida por uma flag.


In [ ]:
from datetime import datetime, timezone

SOURCE = "all"  # all, camara ou senado

DATA_INICIO_VALIDACAO = "2026-05-01"
DATA_FIM_VALIDACAO = "2026-05-18"
SAMPLE_LIMIT_VALIDACAO = 5

DATA_INICIO_PROD = "2011-05-18"
DATA_FIM_PROD = "2026-05-18"

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID_VALIDACAO = f"colab-parlamentares-validacao-{RUN_STAMP}"
RUN_ID_PROCESSAMENTO_VALIDACAO = f"processed-parlamentares-v1-validacao-{RUN_STAMP}"
RUN_ID_JOIN_VALIDACAO = f"parlamentares-join-validacao-{RUN_STAMP}"

RUN_ID_PROD = "prod-parlamentares-baseline"
RUN_ID_PROCESSAMENTO_PROD = "processed-parlamentares-v1-baseline"
RUN_ID_JOIN_PROD = "parlamentares-join-baseline"

print("RUN_ID_VALIDACAO=", RUN_ID_VALIDACAO)
print("RUN_ID_PROCESSAMENTO_VALIDACAO=", RUN_ID_PROCESSAMENTO_VALIDACAO)
print("RUN_ID_PROD=", RUN_ID_PROD, "# fixo para permitir retomada com --resume")


In [ ]:
import importlib.util
import json
import subprocess
import sys
from pathlib import Path

COLLECT_MODULE = "coleta.parlamentares.collect"
PROCESS_MODULE = "processamento.parlamentares"
JOIN_AUDIT_MODULE = "processamento.parlamentares_join_audit"


def module_available(module):
    return importlib.util.find_spec(module) is not None


def run_python_module(module, args):
    cmd = [sys.executable, "-m", module, *args]
    print("Rodando:", " ".join(cmd))
    output_lines = []
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        output_lines.append(line.rstrip("\n"))

    return process.wait(), output_lines


def run_collector(run_id, data_inicio, data_fim, source="all", sample=False, sample_limit=None, resume=True):
    if not module_available(COLLECT_MODULE):
        print("Modulo ainda nao encontrado:", COLLECT_MODULE)
        print("Execute esta celula depois que o coletor de parlamentares estiver implementado no repo.")
        return None

    args = [
        "--mode",
        "prod",
        "--run-id",
        run_id,
        "--data-inicio",
        data_inicio,
        "--data-fim",
        data_fim,
        "--source",
        source,
    ]
    if resume:
        args.append("--resume")
    if sample:
        args.append("--sample")
    if sample_limit is not None:
        args.extend(["--sample-limit", str(sample_limit)])

    returncode, output_lines = run_python_module(COLLECT_MODULE, args)
    manifest_path = manifest_from_output(output_lines, run_id)
    if returncode != 0:
        print(f"Coletor terminou com returncode={returncode}; veja logs/autosave.")
    return manifest_path


def run_processing(run_id, overwrite=True):
    if not module_available(PROCESS_MODULE):
        print("Modulo ainda nao encontrado:", PROCESS_MODULE)
        print("Execute esta celula depois que o processamento de parlamentares estiver implementado no repo.")
        return None

    args = ["--mode", "prod", "--run-id", run_id]
    if overwrite:
        args.append("--overwrite")
    returncode, output_lines = run_python_module(PROCESS_MODULE, args)
    manifest_path = manifest_from_output(output_lines, f"{run_id}-parlamentares")
    if returncode != 0:
        print(f"Processamento terminou com returncode={returncode}; veja logs/autosave.")
    return manifest_path


def run_join_audit(run_id, profile="colab", overwrite=True):
    if not module_available(JOIN_AUDIT_MODULE):
        print("Modulo ainda nao encontrado:", JOIN_AUDIT_MODULE)
        print("Execute esta celula depois que a auditoria de juncao estiver implementada no repo.")
        return None

    args = ["--profile", profile, "--run-id", run_id]
    if overwrite:
        args.append("--overwrite")
    returncode, output_lines = run_python_module(JOIN_AUDIT_MODULE, args)
    if returncode != 0:
        print(f"Auditoria terminou com returncode={returncode}; veja o diretorio de audits.")
    return output_lines


def manifest_from_output(output_lines, run_id):
    for line in reversed(output_lines):
        candidate = line.strip()
        if candidate.endswith(".json"):
            path = Path(candidate)
            if path.exists():
                return path

    manifest_path = DATA_ROOT / "manifests" / f"{run_id}.json"
    autosave_path = DATA_ROOT / "manifests" / f"{run_id}.autosave.json"
    if manifest_path.exists():
        return manifest_path
    if autosave_path.exists():
        return autosave_path
    return None


def show_manifest(manifest_path):
    if manifest_path is None:
        print("Manifest nao foi informado pelo comando.")
        return None

    manifest_path = Path(manifest_path)
    if not manifest_path.exists():
        print("Manifest nao encontrado:", manifest_path)
        return None

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    keys = [
        "run_id",
        "source",
        "dataset",
        "mode",
        "sample",
        "sample_limit",
        "status",
        "errors",
        "record_counts",
        "partition_counts",
        "raw_run_ids",
        "output_records",
        "log_path",
        "autosave_path",
    ]
    resumo = {key: manifest.get(key) for key in keys}
    print(json.dumps(resumo, ensure_ascii=False, indent=2))

    log_path_value = manifest.get("log_path")
    if log_path_value:
        log_path = Path(log_path_value)
        if log_path.exists():
            print("\nUltimas linhas do log:")
            print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-10:]))
    return manifest


def tail_log_for_run(run_id, n=30):
    log_path = DATA_ROOT / "logs" / f"{run_id}.jsonl"
    print("log_path=", log_path)
    if not log_path.exists():
        print("Log ainda nao encontrado.")
        return
    print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-n:]))


## Validacao curta

Esta celula roda em `--mode prod`, mas passa `--sample` para limitar poucos parlamentares por casa. Ela confirma que o Drive esta montado e que a escrita esta indo para `FALANDO_NELA_DATA_ROOT`.


In [ ]:
manifest_validacao_path = run_collector(
    RUN_ID_VALIDACAO,
    DATA_INICIO_VALIDACAO,
    DATA_FIM_VALIDACAO,
    source=SOURCE,
    sample=True,
    sample_limit=SAMPLE_LIMIT_VALIDACAO,
    resume=True,
)
manifest_validacao = show_manifest(manifest_validacao_path)


In [ ]:
def inspect_jsonl(path, max_rows=5):
    path = Path(path)
    print("\nArquivo:", path)
    if not path.exists():
        print("Nao encontrado")
        return

    lines = path.read_text(encoding="utf-8").splitlines()
    print("linhas=", len(lines), "bytes=", path.stat().st_size)
    for line in lines[:max_rows]:
        record = json.loads(line)
        payload = record.get("payload", {})
        payload_keys = sorted(payload.keys())[:8] if isinstance(payload, dict) else []
        print({
            "source": record.get("source"),
            "dataset": record.get("dataset"),
            "record_type": record.get("record_type"),
            "partition": record.get("partition"),
            "source_id": record.get("source_id"),
            "status_code": record.get("response", {}).get("status_code"),
            "payload_keys": payload_keys,
        })


for source in ["camara", "senado"]:
    metadata_path = DATA_ROOT / "raw" / source / "parlamentares" / "metadata" / f"{RUN_ID_VALIDACAO}.jsonl"
    inspect_jsonl(metadata_path, max_rows=5)


## Processamento e auditoria local no Drive

Depois que a coleta curta gerar dados brutos, rode o processamento de `parlamentares/v1`. A auditoria de juncao depende dos Parquets de `textos_parlamentares/v1` ja existirem no Drive.


In [ ]:
manifest_processamento_validacao_path = run_processing(
    RUN_ID_PROCESSAMENTO_VALIDACAO,
    overwrite=True,
)
manifest_processamento_validacao = show_manifest(manifest_processamento_validacao_path)


In [ ]:
EXECUTAR_AUDITORIA_VALIDACAO = False

if EXECUTAR_AUDITORIA_VALIDACAO:
    run_join_audit(RUN_ID_JOIN_VALIDACAO, profile="colab", overwrite=True)
else:
    print("Altere EXECUTAR_AUDITORIA_VALIDACAO para True quando os Parquets de textos e parlamentares existirem no Drive.")


## Coleta completa

Esta celula roda o baseline inteiro em modo producao, com `run_id` fixo e `--resume`. Se o Colab cair ou a API falhar em alguma parte, rode a mesma celula novamente: o coletor le o checkpoint e os JSONLs ja gravados antes de continuar.


In [ ]:
EXECUTAR_COMPLETA = False

if EXECUTAR_COMPLETA:
    print("Rodando coleta completa com run_id fixo:", RUN_ID_PROD)
    manifest_prod_path = run_collector(
        RUN_ID_PROD,
        DATA_INICIO_PROD,
        DATA_FIM_PROD,
        source=SOURCE,
        sample=False,
        resume=True,
    )
    manifest_prod = show_manifest(manifest_prod_path)

    manifest_processamento_prod_path = run_processing(
        RUN_ID_PROCESSAMENTO_PROD,
        overwrite=True,
    )
    manifest_processamento_prod = show_manifest(manifest_processamento_prod_path)
else:
    print("Altere EXECUTAR_COMPLETA para True quando quiser iniciar a coleta completa.")


## Auditoria de juncao completa

Rode esta etapa depois que os Parquets de `textos_parlamentares/v1` e `parlamentares/v1` tiverem sido gerados no Drive.


In [ ]:
EXECUTAR_AUDITORIA_PROD = False

if EXECUTAR_AUDITORIA_PROD:
    run_join_audit(RUN_ID_JOIN_PROD, profile="colab", overwrite=True)
else:
    print("Altere EXECUTAR_AUDITORIA_PROD para True depois de gerar os Parquets completos.")


## Consultar execucao em andamento

Use esta celula em outra sessao do Colab para consultar manifesto, autosave e ultimas linhas de log do `run_id` fixo.


In [ ]:
RUN_ID_MONITORADO = RUN_ID_PROD

autosave_path = DATA_ROOT / "manifests" / f"{RUN_ID_MONITORADO}.autosave.json"
manifest_path = DATA_ROOT / "manifests" / f"{RUN_ID_MONITORADO}.json"

if manifest_path.exists():
    show_manifest(manifest_path)
elif autosave_path.exists():
    show_manifest(autosave_path)
else:
    print("Ainda nao ha manifest/autosave para", RUN_ID_MONITORADO)

tail_log_for_run(RUN_ID_MONITORADO, n=30)
